# 04 — Threshold Optimisation & Business Impact

**Goal:** Move beyond ML metrics to find the operating point that minimises real-world cost.

**Cost model:**

| Event | Cost | Reasoning |
|---|---|---|
| False Negative (missed fraud) | $500 | Chargeback + investigation + reputational damage |
| False Positive (blocked legit tx) | $5 | Customer friction, potential churn |

> The 100:1 asymmetry means recall is far more valuable than precision — but not infinitely so. The optimal threshold balances both.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from data_loader import load_engineered, get_split, TARGET
from metrics import (
    evaluate_model, expected_cost, find_best_threshold,
    DEFAULT_COST_FN, DEFAULT_COST_FP
)

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Load saved model & test data

In [ ]:
pipeline = joblib.load('../models/best_pipeline.pkl')
df = load_engineered()
_, X_test, _, y_test = get_split(df)

proba = pipeline.predict_proba(X_test)[:, 1]
print(f'Test size: {len(y_test):,} | Fraud rate: {y_test.mean():.3%}')

## 2. Threshold sweep — precision, recall, cost

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
records = []

for t in thresholds:
    m = evaluate_model(y_test, proba, threshold=t)
    records.append({
        'threshold': t,
        'precision': m['precision'],
        'recall':    m['recall'],
        'cost':      expected_cost(y_test, proba, t),
    })

sweep = pd.DataFrame(records)
sweep.head()

In [ ]:
best_t, best_cost = find_best_threshold(y_test, proba)
print(f'Optimal threshold : {best_t:.3f}')
print(f'Minimum cost      : ${best_cost:,.0f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Precision & Recall vs threshold
ax = axes[0]
ax.plot(sweep['threshold'], sweep['precision'], label='Precision', color='steelblue', lw=1.8)
ax.plot(sweep['threshold'], sweep['recall'],    label='Recall',    color='tomato',    lw=1.8)
ax.axvline(best_t, color='black', linestyle='--', lw=1, label=f'Optimal t={best_t:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision & Recall vs threshold')
ax.legend()

# Right: Expected cost vs threshold
ax = axes[1]
ax.plot(sweep['threshold'], sweep['cost'], color='darkorange', lw=1.8)
ax.axvline(best_t, color='black', linestyle='--', lw=1, label=f'Optimal t={best_t:.2f}')
ax.scatter([best_t], [best_cost], color='black', zorder=5, s=60)
ax.set_xlabel('Threshold')
ax.set_ylabel('Expected cost ($)')
ax.set_title('Expected cost vs threshold')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/04_threshold_sweep.png', bbox_inches='tight')
plt.show()

## 3. Evaluate at key thresholds

In [ ]:
thresholds_to_compare = {
    'Conservative (t=0.2) — max recall':  0.2,
    'Default (t=0.5)':                    0.5,
    f'Optimal (t={best_t:.2f}) — min cost': best_t,
    'Aggressive (t=0.8) — max precision': 0.8,
}

rows = []
for label, t in thresholds_to_compare.items():
    m = evaluate_model(y_test, proba, threshold=t)
    c = expected_cost(y_test, proba, t)
    rows.append({
        'Scenario':  label,
        'Threshold': round(t, 3),
        'Precision': round(m['precision'], 3),
        'Recall':    round(m['recall'], 3),
        'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'],
        'Cost ($)':  int(c),
    })

comparison = pd.DataFrame(rows).set_index('Scenario')
comparison

## 4. Cost sensitivity analysis — what if our cost assumptions are wrong?

In [ ]:
fn_costs  = [100, 250, 500, 1000, 2000]
fp_costs  = [1, 5, 10, 25]

sensitivity_rows = []
for cost_fn in fn_costs:
    for cost_fp in fp_costs:
        t_opt, c_opt = find_best_threshold(y_test, proba, cost_fn=cost_fn, cost_fp=cost_fp)
        sensitivity_rows.append({
            'cost_fn': cost_fn,
            'cost_fp': cost_fp,
            'optimal_threshold': round(t_opt, 3),
            'optimal_cost': int(c_opt),
        })

sens_df = pd.DataFrame(sensitivity_rows)

# Pivot: rows = FN cost, cols = FP cost
pivot = sens_df.pivot(index='cost_fn', columns='cost_fp', values='optimal_threshold')

import seaborn as sns
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd_r', ax=ax)
ax.set_title('Optimal threshold for different FN / FP cost assumptions')
ax.set_xlabel('FP cost ($)')
ax.set_ylabel('FN cost ($)')
plt.tight_layout()
plt.savefig('../reports/figures/04_cost_sensitivity.png', bbox_inches='tight')
plt.show()

## Threshold analysis summary

- The optimal threshold is significantly lower than the default 0.5, reflecting the high cost of missed fraud vs false positives.
- The sensitivity heatmap shows how the optimal threshold shifts as cost assumptions change — it remains in a low-to-mid range across all plausible scenarios, confirming the robustness of the choice.
- In production, the cost model should be revisited quarterly as chargeback rates and operational costs change.